# Notebook OOP-3 — Contracts & plugins — payments, chargers, smart-home hubs

| | |
|---|---|
| **Focus** | Object-oriented Python (Advanced) |
| **Daily-life theme** | cash vs tap-to-pay, USB charging bars, Alexa-style plugins |
| **Pattern** | Story → Concept → Demo → **Exercise** → Solution |

**Prerequisites:** `02-oop-intermediate-house-rules.ipynb`

**Next notebook:** `exercises/08-advanced-typing-protocols-dataclasses.ipynb`

---

## Learning objectives

- Use `@dataclass(frozen=True)` for immutable IDs.
- Combine **ABC** + subclasses like interchangeable checkout lanes.
- Use **`typing.Protocol`** for "anything that charges phones".
- Recognize dunder hooks (`__repr__`) as readable receipts.

## Table of contents

1. **Frozen dataclass — library card**
2. **ABC checkout lanes**
3. **Protocol charging kiosk**
4. **`__repr__` receipts**
5. **Exercise — smart-home toggle hub**

---

> **Tip:** When an analogy clicks, rename classes to AI-flavored names (`Embedder`, `ToolRegistry`) — same mechanics.

---


## 1 · Frozen dataclass — library card barcode

**Daily life:** Plastic library card **number shouldn't mutate** after issuance—perfect match for `frozen=True`.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class LibraryCard:
    holder: str
    barcode: str


card = LibraryCard("Taylor", "5901234567890")
print(card)
try:
    card.barcode = "nope"
except Exception as e:
    print(type(e).__name__)


## 2 · ABC — checkout accepts cash or tap-to-pay

**Daily life:** Cashier cares that payment **settles**, not whether bills or NFC chip—**abstract interface**, concrete behaviors.


In [ ]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def settle(self, amount_cents: int) -> str:
        ...


class CashPayment(PaymentMethod):
    def settle(self, amount_cents: int) -> str:
        return f"counted ${amount_cents/100:.2f} cash"


class TapPayment(PaymentMethod):
    def settle(self, amount_cents: int) -> str:
        return f"tapped NFC ${amount_cents/100:.2f}"


def checkout(method: PaymentMethod, cents: int) -> None:
    print(method.settle(cents))

checkout(CashPayment(), 425)
checkout(TapPayment(), 425)


## 3 · Protocol — charging kiosk accepts mystery gadgets

**Daily life:** Airport kiosk doesn't ask "what brand phone?" — only checks "does **Chargeable** plug work?" (`Protocol`).


In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Chargeable(Protocol):
    def charge_minutes(self, minutes: int) -> int:
        ...


class Phone:
    def charge_minutes(self, minutes: int) -> int:
        return minutes * 2  # fake percent gain


class Laptop:
    def charge_minutes(self, minutes: int) -> int:
        return minutes


def kiosk_boost(device: Chargeable, minutes: int) -> None:
    gain = device.charge_minutes(minutes)
    print(f"kiosk delivered ~{gain}% juice")


kiosk_boost(Phone(), 15)
print(isinstance(Laptop(), Chargeable))


## 4 · `__repr__` — readable grocery receipts

**Daily life:** Receipt lines formatted uniformly — Python uses `__repr__` / dataclass defaults for debugging.


In [ ]:
from dataclasses import dataclass

@dataclass
class CartLine:
    sku: str
    qty: int
    unit_usd: float

    def total(self) -> float:
        return self.qty * self.unit_usd


print([CartLine("milk", 2, 3.5), CartLine("bread", 1, 4.25)])


### Exercise — smart-home hub (`Protocol`)

**Daily life:** Wall tablet toggles lamps **without knowing vendor firmware** — protocol talks "toggle" only.

Requirements:

1. Define `@runtime_checkable` protocol `ToggleDevice` with `label() -> str` and `toggle() -> None`.
2. Classes `DeskLamp` and `Fan` implement it (track `_on` bool).
3. `SmartHub.toggle_all(devices)` prints each label then toggles.


In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class ToggleDevice(Protocol):
    def label(self) -> str: ...
    def toggle(self) -> None: ...

class DeskLamp:
    def __init__(self, name: str) -> None:
        raise NotImplementedError

    def label(self) -> str:
        raise NotImplementedError

    def toggle(self) -> None:
        raise NotImplementedError


class Fan:
    def __init__(self, name: str) -> None:
        raise NotImplementedError

    def label(self) -> str:
        raise NotImplementedError

    def toggle(self) -> None:
        raise NotImplementedError


class SmartHub:
    def toggle_all(self, devices: list[ToggleDevice]) -> None:
        raise NotImplementedError


# After implementing, uncomment:
# hub = SmartHub()
# hub.toggle_all([DeskLamp("Reading"), Fan("Ceiling")])
print("Implement classes, then uncomment the demo.")


### Solution — smart-home hub

<details>
<summary>Reveal solution</summary>

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class ToggleDevice(Protocol):
    def label(self) -> str: ...
    def toggle(self) -> None: ...

class DeskLamp:
    def __init__(self, name: str) -> None:
        self._name = name
        self._on = False

    def label(self) -> str:
        return f"Lamp:{self._name}"

    def toggle(self) -> None:
        self._on = not self._on


class Fan:
    def __init__(self, name: str) -> None:
        self._name = name
        self._on = False

    def label(self) -> str:
        return f"Fan:{self._name}"

    def toggle(self) -> None:
        self._on = not self._on


class SmartHub:
    def toggle_all(self, devices: list[ToggleDevice]) -> None:
        for d in devices:
            print(d.label())
            d.toggle()
```

</details>
